# 🔥 Train mô hình phát hiện Khói & Lửa sớm (Smoke & Fire YOLOv8n Detector)

Notebook này huấn luyện mô hình **YOLOv8n Object Detection** chuyên biệt cho bài toán **Phát hiện sớm Khói và Lửa (Smoke & Fire Detection)** kết hợp phân tích hành vi con người châm lửa trong hệ thống giám sát thông minh.

- **Kiến trúc:** `YOLOv8n` (3 tầng đặc trưng P3/P4/P5 ở độ phân giải `640×640`), bắt được cả **đốm lửa nhỏ từ bật lửa / diêm / tàn thuốc** trên tay người lẫn **đám khói / đám cháy sớm** ở không gian rộng và ngoài trời.
- **Nhãn phân loại (2 class):** `0: fire` (Lửa / Ngọn lửa), `1: smoke` (Khói).
- **Dữ liệu (Tự động tải 100% từ Hugging Face & GitHub, không cần tài khoản):**
  1. **Indoor & Close-up Fire/Smoke Dataset** (~5.000 ảnh, ~200 MB): Lửa và khói trong nhà, ngọn lửa nhỏ (bật lửa, nến, chập điện) và đám cháy.
  2. **CCTV Smoke & Fire Emergency Dataset** (~240 ảnh góc rộng camera giám sát, ~339 MB): Khói và lửa thực tế từ góc nhìn camera an ninh.
  3. **COCO128 Hard Negatives** (~128 ảnh đa dạng không có lửa/khói): Đèn giao thông, người mặc áo đỏ/vàng, phòng ốc bình thường để **chống báo động giả (False Positive)**.
- **Kết quả đầu ra:** `fire_smoke_yolov8n.pt` (chép vào thư mục `weights/` của dự án) cùng 2 biểu đồ `fire_smoke_training_curves.png` và `fire_smoke_confusion_matrix.png`.

**Cách dùng:** trên Google Colab chọn *Runtime → Change runtime type → T4 GPU*, sau đó bấm *Runtime → Run all*. Toàn bộ quá trình mất khoảng **20–25 phút**.

In [ ]:
# 1. Cài đặt thư viện Ultralytics YOLOv8 và kiểm tra GPU
!pip install -q ultralytics opencv-python-headless matplotlib pyyaml

import torch, cv2, sys, ultralytics
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ KHÔNG CÓ GPU - Hãy bật T4 GPU trong Runtime > Change runtime type!")

## 1. Cấu hình huấn luyện

Các siêu tham số dưới đây đã được tinh chỉnh riêng cho đặc trưng vật lý của **Lửa (đỏ-cam-vàng)** và **Khói (xám-trắng-đen)**:
- Giữ `hsv_h = 0.01` rất nhỏ để không làm biến đổi màu đỏ/cam của ngọn lửa thành màu xanh/tím khi tăng cường dữ liệu.
- Tắt `flipud = 0.0` (không lật ngược ảnh trên-dưới) vì ngọn lửa và cột khói tự nhiên luôn bốc lên phía trên.
- Bật `mosaic = 1.0` và `scale = 0.6` để tăng khả năng phát hiện **đốm lửa nhỏ** (khi bật quẹt gas trước webcam hoặc khi lửa mới nhen nhóm từ xa).

In [ ]:
SEED = 42
MODEL_BASE = "yolov8n.pt"   # Có thể đổi thành "yolov8s.pt" nếu muốn ưu tiên độ chính xác cao hơn nữa
EPOCHS = 35                 # 35 epoch trên ~5.300 ảnh hội tụ rất tốt trên GPU T4
BATCH_SIZE = 32             # Vừa vặn VRAM 15GB của Colab T4
IMGSZ = 640                 # Độ phân giải chuẩn khớp với pipeline dự án

USE_INDOOR_FIRE_SMOKE = True   # ~5.000 ảnh lửa/khói trong nhà & lửa nhỏ (200 MB)
USE_CCTV_EMERGENCY = True      # ~240 ảnh camera giám sát góc rộng/ngoài trời (339 MB)
ADD_HARD_NEGATIVES = True      # Thêm ảnh nền không có lửa/khói (áo đỏ, đèn vàng, người) để chống báo giả

MOUNT_DRIVE = False            # Đặt True nếu muốn lưu tự động bản sao trọng số lên Google Drive
SAVE_TO_DRIVE_DIR = ""         # Ví dụ: "/content/drive/MyDrive/btl-httm"

DATASET_ROOT = "/content/fire_smoke_dataset"

## 2. Tự động tải và chuẩn hóa bộ dữ liệu

Ô code dưới đây tự động tải các gói dữ liệu từ Hugging Face và GitHub, lọc bỏ file rác (`__MACOSX`), đồng nhất nhãn `0: fire` và `1: smoke`, chia tập `train / valid / test` và bổ sung ảnh âm tính (Hard Negatives).

In [ ]:
import os
import glob
import random
import shutil
import zipfile
import urllib.request
from pathlib import Path

random.seed(SEED)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

os.makedirs("/content/downloads", exist_ok=True)

def download_file(url: str, dest: str):
    if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024:
        print(f"[Đã có sẵn] {os.path.basename(dest)} ({os.path.getsize(dest) / 1e6:.1f} MB)")
        return
    print(f"Đang tải {os.path.basename(dest)}...")
    urllib.request.urlretrieve(url, dest)
    print(f"  -> Hoàn tất: {os.path.getsize(dest) / 1e6:.1f} MB")

# Tạo cấu trúc thư mục chuẩn YOLOv8
for split in ("train", "valid", "test"):
    os.makedirs(f"{DATASET_ROOT}/{split}/images", exist_ok=True)
    os.makedirs(f"{DATASET_ROOT}/{split}/labels", exist_ok=True)

def clean_and_copy_pair(img_path: str, lbl_path: str, split: str, prefix: str) -> bool:
    """Kiểm tra tính hợp lệ của file ảnh và nhãn YOLO (chỉ giữ class 0: fire, 1: smoke)."""
    ext = os.path.splitext(img_path)[1].lower()
    if ext not in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
        return False
    stem = Path(img_path).stem
    out_name = f"{prefix}_{stem}"
    dst_img = f"{DATASET_ROOT}/{split}/images/{out_name}{ext}"
    dst_lbl = f"{DATASET_ROOT}/{split}/labels/{out_name}.txt"

    valid_lines = []
    if lbl_path and os.path.exists(lbl_path):
        with open(lbl_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    try:
                        cls_id = int(float(parts[0]))
                        coords = [float(x) for x in parts[1:5]]
                        if cls_id in (0, 1) and all(0.0 <= c <= 1.0 for c in coords) and coords[2] > 0 and coords[3] > 0:
                            valid_lines.append(f"{cls_id} {coords[0]:.6f} {coords[1]:.6f} {coords[2]:.6f} {coords[3]:.6f}")
                    except ValueError:
                        continue

    shutil.copy2(img_path, dst_img)
    with open(dst_lbl, "w", encoding="utf-8") as f:
        f.write("\n".join(valid_lines) + ("\n" if valid_lines else ""))
    return True

# 1. Tải và gộp bộ Indoor Fire & Smoke (~5000 ảnh)
if USE_INDOOR_FIRE_SMOKE:
    indoor_zip = "/content/downloads/indoor_fire_smoke.zip"
    download_file(
        "https://huggingface.co/datasets/KienNgyuen/Fire-Smoke-Detection/resolve/main/Indoor%20Fire%20Smoke.zip",
        indoor_zip
    )
    extract_dir = "/content/raw_indoor"
    if not os.path.exists(extract_dir):
        with zipfile.ZipFile(indoor_zip, "r") as zf:
            zf.extractall(extract_dir)
    base_dir = os.path.join(extract_dir, "Indoor Fire Smoke")
    count = 0
    for split in ("train", "valid", "test"):
        img_dir = os.path.join(base_dir, split, "images")
        lbl_dir = os.path.join(base_dir, split, "labels")
        if not os.path.isdir(img_dir):
            continue
        for img_file in os.listdir(img_dir):
            if img_file.startswith("._"):
                continue
            img_p = os.path.join(img_dir, img_file)
            lbl_p = os.path.join(lbl_dir, Path(img_file).stem + ".txt")
            if clean_and_copy_pair(img_p, lbl_p, split, "indoor"):
                count += 1
    print(f"Đã nạp {count} ảnh từ Indoor Fire & Smoke Dataset.")

# 2. Tải và gộp bộ CCTV Smoke & Fire Emergency (~240 ảnh góc rộng CCTV)
if USE_CCTV_EMERGENCY:
    cctv_zip = "/content/downloads/cctv_fire_smoke.zip"
    download_file(
        "https://huggingface.co/datasets/KienNgyuen/Fire-Smoke-Detection/resolve/main/CCTV%20Smoke%20%26%20Fire%20Emergency%20Detection%20Dataset.zip",
        cctv_zip
    )
    extract_cctv = "/content/raw_cctv"
    if not os.path.exists(extract_cctv):
        with zipfile.ZipFile(cctv_zip, "r") as zf:
            zf.extractall(extract_cctv)
    img_dir = os.path.join(extract_cctv, "images")
    lbl_dir = os.path.join(extract_cctv, "labels")
    all_imgs = sorted([f for f in os.listdir(img_dir) if not f.startswith("._")])
    random.shuffle(all_imgs)
    n_total = len(all_imgs)
    n_train = int(0.8 * n_total)
    n_val = int(0.1 * n_total)
    cctv_count = 0
    for idx, img_file in enumerate(all_imgs):
        split = "train" if idx < n_train else ("valid" if idx < n_train + n_val else "test")
        img_p = os.path.join(img_dir, img_file)
        lbl_p = os.path.join(lbl_dir, Path(img_file).stem + ".txt")
        if clean_and_copy_pair(img_p, lbl_p, split, "cctv"):
            cctv_count += 1
    print(f"Đã nạp {cctv_count} ảnh từ CCTV Smoke & Fire Emergency Dataset.")

# 3. Thêm ảnh âm tính khó (Hard Negatives từ COCO128 - không có lửa/khói)
if ADD_HARD_NEGATIVES:
    coco_zip = "/content/downloads/coco128.zip"
    download_file("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip", coco_zip)
    extract_coco = "/content/raw_coco128"
    if not os.path.exists(extract_coco):
        with zipfile.ZipFile(coco_zip, "r") as zf:
            zf.extractall(extract_coco)
    coco_imgs = sorted(glob.glob(f"{extract_coco}/coco128/images/train2017/*.jpg"))
    neg_count = 0
    for idx, img_p in enumerate(coco_imgs):
        split = "train" if idx < 100 else ("valid" if idx < 114 else "test")
        # Truyền lbl_path=None để tạo file nhãn rỗng (Background không có lửa/khói)
        if clean_and_copy_pair(img_p, None, split, "neg"):
            neg_count += 1
    print(f"Đã bổ sung {neg_count} ảnh nền âm tính (Hard Negatives) để chống báo động giả.")

## 3. Tạo file cấu hình `fire_smoke.yaml` và thống kê dữ liệu

In [ ]:
import yaml
from collections import Counter

yaml_path = f"{DATASET_ROOT}/fire_smoke.yaml"
data_cfg = {
    "path": DATASET_ROOT,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 2,
    "names": {0: "fire", 1: "smoke"}
}
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_cfg, f, sort_keys=False, allow_unicode=True)

for split in ("train", "valid", "test"):
    imgs = glob.glob(f"{DATASET_ROOT}/{split}/images/*")
    lbls = glob.glob(f"{DATASET_ROOT}/{split}/labels/*.txt")
    cls_counter = Counter()
    empty_lbls = 0
    for lp in lbls:
        with open(lp, "r") as lf:
            lines = [ln.strip() for ln in lf if ln.strip()]
        if not lines:
            empty_lbls += 1
        for ln in lines:
            cls_counter[int(ln.split()[0])] += 1
    print(f"[{split.upper():5s}] Tổng ảnh: {len(imgs):4d} | Box Lửa (0): {cls_counter[0]:4d} | Box Khói (1): {cls_counter[1]:4d} | Ảnh nền âm tính: {empty_lbls:3d}")

## 4. Huấn luyện mô hình YOLOv8n (`fire_smoke_yolov8n.pt`)

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_BASE)
results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else "cpu",
    seed=SEED,
    # Tăng cường dữ liệu tối ưu cho phát hiện sớm Lửa & Khói:
    mosaic=1.0,       # Ghép 4 ảnh giúp học tốt các đốm lửa/khói kích thước nhỏ
    scale=0.6,        # Đa dạng khoảng cách xa/gần so với camera
    hsv_h=0.01,       # Giữ tông màu Đỏ-Cam-Vàng tự nhiên của ngọn lửa
    hsv_s=0.5,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,       # Không lật ngược trên-dưới vì lửa và khói luôn bốc lên trên
    project="/content/runs",
    name="fire_smoke_yolov8n",
    exist_ok=True,
)

## 5. Đánh giá độc lập trên tập Test và xuất biểu đồ báo cáo

Đánh giá trọng số tốt nhất (`best.pt`) trên tập **test** để lấy các chỉ số `Precision`, `Recall`, `mAP@50` và `mAP@50-95` đưa vào Chương 4 của báo cáo.

In [ ]:
import matplotlib.pyplot as plt

run_dir = "/content/runs/fire_smoke_yolov8n"
best_pt = f"{run_dir}/weights/best.pt"

best_model = YOLO(best_pt)
metrics = best_model.val(data=yaml_path, split="test", imgsz=IMGSZ)

print("\n================ KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ================")
print(f"  Precision (chung) : {metrics.box.mp:.4f}")
print(f"  Recall (chung)    : {metrics.box.mr:.4f}")
print(f"  mAP@50 (chung)    : {metrics.box.map50:.4f}")
print(f"  mAP@50-95 (chung) : {metrics.box.map:.4f}")
for i, cls_name in enumerate(["fire (Lửa)", "smoke (Khói)"]):
    if i < len(metrics.box.ap50):
        p, r, ap50, ap = metrics.box.class_result(i)
        print(f"  - Lớp {cls_name:12s}: P={p:.4f} | R={r:.4f} | mAP@50={ap50:.4f} | mAP@50-95={ap:.4f}")
print("================================================================\n")

# Sao chép biểu đồ đường cong huấn luyện và ma trận nhầm lẫn ra thư mục gốc để tải về
if os.path.exists(f"{run_dir}/results.png"):
    shutil.copy2(f"{run_dir}/results.png", "fire_smoke_training_curves.png")
if os.path.exists(f"{run_dir}/confusion_matrix.png"):
    shutil.copy2(f"{run_dir}/confusion_matrix.png", "fire_smoke_confusion_matrix.png")

## 6. Đóng gói `fire_smoke_yolov8n.pt` và tải về máy

**Hướng dẫn đưa vào dự án (`e:\btl-httm`):**
1. Chép file `fire_smoke_yolov8n.pt` vừa tải về vào thư mục `weights/fire_smoke_yolov8n.pt`.
2. Trong `configs/config.yaml`, mục `behaviors.fire_smoke.enabled` đã được bật sẵn (`true`).
3. Chạy `python main.py` và mở `http://localhost:8000` — bạn có thể bật thử chiếc bật lửa trước Webcam hoặc tải video có khói/lửa lên để xem cảnh báo thời gian thực!

In [ ]:
output_weight = "fire_smoke_yolov8n.pt"
shutil.copy2(best_pt, output_weight)
print(f"Đã xuất trọng số: {output_weight} ({os.path.getsize(output_weight) / 1e6:.2f} MB)")

if SAVE_TO_DRIVE_DIR:
    os.makedirs(SAVE_TO_DRIVE_DIR, exist_ok=True)
    for f in (output_weight, "fire_smoke_training_curves.png", "fire_smoke_confusion_matrix.png"):
        if os.path.exists(f):
            shutil.copy2(f, SAVE_TO_DRIVE_DIR)
    print("Đã lưu bản sao lên Google Drive tại:", SAVE_TO_DRIVE_DIR)

try:
    from google.colab import files
    for f in (output_weight, "fire_smoke_training_curves.png", "fire_smoke_confusion_matrix.png"):
        if os.path.exists(f):
            files.download(f)
except ImportError:
    pass